# Model 3 Training: Restormer Transformer for Image Restoration (Local CPU - 20 Epochs)

This notebook trains Model 3: Restormer Transformer locally on CPU for 20 epochs.

### Configuration
- Device: CPU only (no GPU required).
- Dataset: Local path at backend/degraded_dataset with train / val / test splits.
- Output: All results (model checkpoint, metrics JSON, plots, zip) saved to backend/transformer/.
- Batch size is small (4) to keep RAM usage manageable on a standard laptop or desktop.
- Gradient accumulation (8 steps) keeps effective batch at 32 for good convergence.

### Key Implementation Details:
- Architecture: Restormer (MDTA Self-Attention + GDFN Feed-Forward + Skip Connections) with dim=32.
- Cell Execution Timers: Tracks individual cell execution time and cumulative elapsed time.
- Training Duration: 20 epochs with Cosine Annealing Learning Rate scheduling.
- Model Checkpointing: Saves best model based on Validation PSNR to transformer_model.pth.
- Results Export: Packages model, metrics, curves, and sample visualizations into transformer_restoration_results_local.zip.

In [ ]:
import time
GLOBAL_START_TIME = time.time()
_cell_start = time.time()

import os
import gc
import glob
import json
import random
import zipfile
from PIL import Image
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def print_timer(cell_name, cell_start):
    cell_elapsed = time.time() - cell_start
    total_elapsed = time.time() - GLOBAL_START_TIME
    print(f"[Cell Timer - {cell_name}] Cell Time: {cell_elapsed:.2f}s | Cumulative Time: {total_elapsed:.2f}s")

print_timer("Imports and Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Local dataset and output paths
DATASET_ROOT = r"C:\Users\202511026\OneDrive\Desktop\HK\Image Restoration Project\backend\degraded_dataset"
OUTPUT_DIR   = r"C:\Users\202511026\OneDrive\Desktop\HK\Image Restoration Project\backend\transformer"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# CPU-only device (no GPU required)
DEVICE   = torch.device("cpu")
USE_CUDA = False
NUM_GPUS = 0

print(f"Device            : {DEVICE}")
print(f"Dataset Root      : {DATASET_ROOT}")
print(f"Output Directory  : {OUTPUT_DIR}")
print(f"Dataset exists    : {os.path.exists(DATASET_ROOT)}")

print_timer("Path and Device Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Dataset Definition for 7-Combination Paired Degraded Images
class LocalDegradedDataset(Dataset):
    def __init__(self, root_dir, split="train", target_size=(128, 128)):
        self.root_dir    = root_dir
        self.split       = split
        self.target_size = target_size
        self.split_dir   = os.path.join(root_dir, split)
        self.clean_dir   = os.path.join(self.split_dir, "clean")
        self.combos = [
            "c1_noise_lowlight",
            "c2_noise_blur",
            "c3_lowlight_blur",
            "c4_noise_only",
            "c5_lowlight_only",
            "c6_blur_only",
            "c7_all_three"
        ]
        self.pairs = []
        if os.path.exists(self.clean_dir):
            clean_files = sorted(
                glob.glob(os.path.join(self.clean_dir, "*.png")) +
                glob.glob(os.path.join(self.clean_dir, "*.jpg"))
            )
            for clean_path in clean_files:
                filename = os.path.basename(clean_path)
                for combo in self.combos:
                    deg_path = os.path.join(self.split_dir, combo, filename)
                    if os.path.exists(deg_path):
                        self.pairs.append((deg_path, clean_path, combo))
        else:
            print(f"WARNING: Clean directory not found at {self.clean_dir}")
        self.to_tensor = T.Compose([T.Resize(target_size), T.ToTensor()])
        print(f"Loaded [{split.upper()}] dataset split: {len(self.pairs)} paired samples.")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        deg_path, clean_path, combo = self.pairs[idx]
        with Image.open(deg_path) as deg_img, Image.open(clean_path) as clean_img:
            deg_tensor   = self.to_tensor(deg_img.convert("RGB"))
            clean_tensor = self.to_tensor(clean_img.convert("RGB"))
        return deg_tensor, clean_tensor

# Small batch size for CPU RAM safety; gradient accumulation keeps effective batch at 32
PHYSICAL_BATCH = 4
ACCUM_STEPS    = 8   # effective batch = 4 * 8 = 32

train_dataset = LocalDegradedDataset(DATASET_ROOT, split="train")
val_dataset   = LocalDegradedDataset(DATASET_ROOT, split="val")
test_dataset  = LocalDegradedDataset(DATASET_ROOT, split="test")

# num_workers=0 for CPU - avoids multiprocessing overhead on Windows
loader_kwargs = {"num_workers": 0, "pin_memory": False}

train_loader = DataLoader(train_dataset, batch_size=PHYSICAL_BATCH, shuffle=True,  drop_last=True,  **loader_kwargs)
val_loader   = DataLoader(val_dataset,   batch_size=PHYSICAL_BATCH, shuffle=False, drop_last=False, **loader_kwargs)
test_loader  = DataLoader(test_dataset,  batch_size=PHYSICAL_BATCH, shuffle=False, drop_last=False, **loader_kwargs)

print(f"Physical Batch: {PHYSICAL_BATCH} | Grad Accumulation Steps: {ACCUM_STEPS} | Effective Batch: {PHYSICAL_BATCH * ACCUM_STEPS}")
print(f"Dataset Pair Counts -> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print_timer("Dataset and DataLoaders Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Model 3: Restormer Transformer (CPU-compatible, dim=32 for lower memory usage)
class MDTA(nn.Module):
    """Multi-Dconv Head Transposed Self-Attention (MDTA) module."""
    def __init__(self, dim, num_heads=4, bias=False):
        super(MDTA, self).__init__()
        self.num_heads   = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))
        self.qkv         = nn.Conv2d(dim, dim * 3, kernel_size=1, bias=bias)
        self.qkv_dwconv  = nn.Conv2d(dim * 3, dim * 3, kernel_size=3, stride=1, padding=1, groups=dim * 3, bias=bias)
        self.project_out = nn.Conv2d(dim, dim, kernel_size=1, bias=bias)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv        = self.qkv_dwconv(self.qkv(x))
        q, k, v    = qkv.chunk(3, dim=1)
        q = q.reshape(b, self.num_heads, -1, h * w)
        k = k.reshape(b, self.num_heads, -1, h * w)
        v = v.reshape(b, self.num_heads, -1, h * w)
        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)
        attn = (q @ k.transpose(-2, -1)) * self.temperature
        attn = attn.softmax(dim=-1)
        out  = (attn @ v).reshape(b, c, h, w)
        return self.project_out(out)

class GDFN(nn.Module):
    """Gated-Dconv Feed-Forward Network (GDFN) module."""
    def __init__(self, dim, ffn_expansion_factor=2.66, bias=False):
        super(GDFN, self).__init__()
        hidden = int(dim * ffn_expansion_factor)
        self.project_in  = nn.Conv2d(dim, hidden * 2, kernel_size=1, bias=bias)
        self.dwconv      = nn.Conv2d(hidden * 2, hidden * 2, kernel_size=3, stride=1, padding=1, groups=hidden * 2, bias=bias)
        self.project_out = nn.Conv2d(hidden, dim, kernel_size=1, bias=bias)

    def forward(self, x):
        x       = self.project_in(x)
        x1, x2  = self.dwconv(x).chunk(2, dim=1)
        return self.project_out(F.gelu(x1) * x2)

class LayerNorm2d(nn.Module):
    def __init__(self, channels, eps=1e-6):
        super(LayerNorm2d, self).__init__()
        self.weight = nn.Parameter(torch.ones(1, channels, 1, 1))
        self.bias   = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.eps    = eps

    def forward(self, x):
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        return self.weight * x + self.bias

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=4, ffn_expansion_factor=2.66, bias=False):
        super(TransformerBlock, self).__init__()
        self.norm1 = LayerNorm2d(dim)
        self.attn  = MDTA(dim, num_heads, bias)
        self.norm2 = LayerNorm2d(dim)
        self.ffn   = GDFN(dim, ffn_expansion_factor, bias)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class Restormer(nn.Module):
    """
    Model 3: Restormer Transformer (CPU local version, dim=32).
    """
    def __init__(self, in_channels=3, out_channels=3, dim=32, num_blocks=[2, 2, 2, 2], heads=[1, 2, 4, 4]):
        super(Restormer, self).__init__()
        self.patch_embed    = nn.Conv2d(in_channels, dim, kernel_size=3, padding=1)
        self.encoder_level1 = nn.Sequential(*[TransformerBlock(dim=dim,     num_heads=heads[0]) for _ in range(num_blocks[0])])
        self.down1_2        = nn.Conv2d(dim, dim * 2, kernel_size=4, stride=2, padding=1)
        self.encoder_level2 = nn.Sequential(*[TransformerBlock(dim=dim * 2, num_heads=heads[1]) for _ in range(num_blocks[1])])
        self.bottleneck     = nn.Sequential(*[TransformerBlock(dim=dim * 2, num_heads=heads[2]) for _ in range(num_blocks[2])])
        self.up2_1          = nn.ConvTranspose2d(dim * 2, dim, kernel_size=4, stride=2, padding=1)
        self.reduce_chan    = nn.Conv2d(dim * 2, dim, kernel_size=1)
        self.decoder_level1 = nn.Sequential(*[TransformerBlock(dim=dim,     num_heads=heads[3]) for _ in range(num_blocks[3])])
        self.output         = nn.Sequential(nn.Conv2d(dim, out_channels, kernel_size=3, padding=1), nn.Sigmoid())

    def forward(self, inp_img):
        feat  = self.patch_embed(inp_img)
        e1    = self.encoder_level1(feat)
        e2    = self.encoder_level2(self.down1_2(e1))
        b     = self.bottleneck(e2)
        u1    = self.reduce_chan(torch.cat([self.up2_1(b), e1], dim=1))
        d_out = self.decoder_level1(u1)
        return self.output(d_out + feat)

model = Restormer().to(DEVICE)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Restormer Model initialized on CPU. Total Trainable Parameters: {num_params:,}")
print_timer("Model Setup", _cell_start)

In [ ]:
_cell_start = time.time()

class CombinedLoss(nn.Module):
    def __init__(self, l1_weight=0.8, mse_weight=0.2):
        super(CombinedLoss, self).__init__()
        self.l1_weight  = l1_weight
        self.mse_weight = mse_weight
        self.l1  = nn.L1Loss()
        self.mse = nn.MSELoss()

    def forward(self, pred, target):
        return self.l1_weight * self.l1(pred, target) + self.mse_weight * self.mse(pred, target)

def calculate_batch_psnr(pred, target):
    with torch.no_grad():
        mse  = torch.mean((pred - target) ** 2, dim=[1, 2, 3])
        psnr = 10.0 * torch.log10(1.0 / (mse + 1e-8))
        return torch.mean(psnr).item()

def calculate_batch_ssim(pred, target):
    with torch.no_grad():
        mu_x     = F.avg_pool2d(pred,   3, 1, 1)
        mu_y     = F.avg_pool2d(target, 3, 1, 1)
        sigma_x  = F.avg_pool2d(pred * pred,     3, 1, 1) - mu_x ** 2
        sigma_y  = F.avg_pool2d(target * target, 3, 1, 1) - mu_y ** 2
        sigma_xy = F.avg_pool2d(pred * target,   3, 1, 1) - mu_x * mu_y
        C1, C2  = 0.01 ** 2, 0.03 ** 2
        ssim_map = ((2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)) / \
                   ((mu_x ** 2 + mu_y ** 2 + C1) * (sigma_x + sigma_y + C2))
        return torch.mean(ssim_map).item()

def extract_state_dict(m):
    return m.state_dict()

criterion = CombinedLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.999), weight_decay=1e-5)

EPOCHS    = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print(f"Epochs: {EPOCHS} | Optimizer: Adam lr=1e-4 | Scheduler: CosineAnnealing")
print(f"Gradient Accumulation: {ACCUM_STEPS} steps | Effective Batch: {PHYSICAL_BATCH * ACCUM_STEPS}")
print_timer("Loss and Metric Functions Setup", _cell_start)

In [ ]:
_cell_start = time.time()

MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "transformer_model.pth")
best_val_psnr   = 0.0

history = {
    "train_loss": [],
    "val_loss":   [],
    "val_psnr":   [],
    "val_ssim":   [],
    "lr":         []
}

print(f"Starting Local CPU Training for {EPOCHS} Epochs...")
print("Note: CPU training is slow. Each epoch may take several hours depending on dataset size.")

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    # ---- Training Phase ----
    model.train()
    train_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    train_pbar = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Train]",
        leave=False,
        mininterval=5.0
    )

    for step_idx, (degraded, clean) in train_pbar:
        output = model(degraded)
        loss   = criterion(output, clean) / ACCUM_STEPS
        loss.backward()

        if (step_idx + 1) % ACCUM_STEPS == 0 or (step_idx + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        batch_loss  = loss.item() * ACCUM_STEPS
        train_loss += batch_loss
        train_pbar.set_postfix({"Loss": f"{batch_loss:.4f}"})
        del degraded, clean, output, loss

    train_loss  /= max(len(train_loader), 1)
    current_lr   = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(round(train_loss, 4))
    history["lr"].append(round(current_lr, 8))

    # ---- Validation Phase ----
    model.eval()
    val_loss = 0.0
    val_psnr = 0.0
    val_ssim = 0.0

    val_pbar = tqdm(
        val_loader,
        desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Val]  ",
        leave=False,
        mininterval=5.0
    )

    with torch.no_grad():
        for degraded, clean in val_pbar:
            restored  = model(degraded)
            loss      = criterion(restored, clean)
            val_loss += loss.item()
            val_psnr += calculate_batch_psnr(restored, clean)
            val_ssim += calculate_batch_ssim(restored, clean)
            del degraded, clean, restored, loss

    val_loss /= max(len(val_loader), 1)
    val_psnr /= max(len(val_loader), 1)
    val_ssim /= max(len(val_loader), 1)

    history["val_loss"].append(round(val_loss, 4))
    history["val_psnr"].append(round(val_psnr, 2))
    history["val_ssim"].append(round(val_ssim, 4))

    scheduler.step()

    epoch_time = time.time() - epoch_start
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} [{epoch_time:.1f}s] -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val PSNR: {val_psnr:.2f} dB | Val SSIM: {val_ssim:.4f} | LR: {current_lr:.2e}")

    if val_psnr > best_val_psnr:
        best_val_psnr = val_psnr
        torch.save(extract_state_dict(model), MODEL_SAVE_PATH)
        print(f"  --> Saved best checkpoint to '{MODEL_SAVE_PATH}' (Val PSNR: {val_psnr:.2f} dB)")

    gc.collect()

print_timer("20-Epoch Local CPU Training Loop", _cell_start)

In [ ]:
_cell_start = time.time()

# Load best checkpoint and run test evaluation
if os.path.exists(MODEL_SAVE_PATH):
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
    print(f"Loaded best checkpoint from {MODEL_SAVE_PATH}")

model.eval()
deg_psnr_total, deg_ssim_total = 0.0, 0.0
res_psnr_total, res_ssim_total = 0.0, 0.0

with torch.no_grad():
    for degraded, clean in tqdm(test_loader, desc="Evaluating Test Set", mininterval=2.0):
        restored        = model(degraded)
        deg_psnr_total += calculate_batch_psnr(degraded, clean)
        deg_ssim_total += calculate_batch_ssim(degraded, clean)
        res_psnr_total += calculate_batch_psnr(restored, clean)
        res_ssim_total += calculate_batch_ssim(restored, clean)
        del degraded, clean, restored

n_test = max(len(test_loader), 1)
avg_deg_psnr = deg_psnr_total / n_test
avg_deg_ssim = deg_ssim_total / n_test
avg_res_psnr = res_psnr_total / n_test
avg_res_ssim = res_ssim_total / n_test
psnr_gain    = avg_res_psnr - avg_deg_psnr
ssim_gain    = avg_res_ssim - avg_deg_ssim

print("\n======================================================")
print("MODEL 3 (RESTORMER TRANSFORMER - LOCAL CPU - 20 EPOCHS)")
print("======================================================")
print(f"Device                  : CPU")
print(f"Degraded Input Quality  : PSNR = {avg_deg_psnr:.2f} dB | SSIM = {avg_deg_ssim:.4f}")
print(f"Transformer Restored    : PSNR = {avg_res_psnr:.2f} dB | SSIM = {avg_res_ssim:.4f}")
print(f"Restoration dB Gain     : PSNR Gain = +{psnr_gain:.2f} dB | SSIM Gain = +{ssim_gain:.4f}")
print("======================================================")

metrics_json_path = os.path.join(OUTPUT_DIR, "training_metrics_local.json")
metrics_data = {
    "model_name": "Restormer Transformer Restoration Network (Local CPU, 20 Epochs)",
    "device": "cpu",
    "epochs": EPOCHS,
    "physical_batch_size": PHYSICAL_BATCH,
    "gradient_accumulation_steps": ACCUM_STEPS,
    "effective_batch_size": PHYSICAL_BATCH * ACCUM_STEPS,
    "loss_functions": ["L1Loss", "MSELoss"],
    "optimizer": "Adam (lr=1e-4, CosineAnnealing)",
    "history": history,
    "test_results": {
        "degraded_psnr": round(avg_deg_psnr, 2),
        "degraded_ssim": round(avg_deg_ssim, 4),
        "restored_psnr": round(avg_res_psnr, 2),
        "restored_ssim": round(avg_res_ssim, 4),
        "psnr_gain_db":  round(psnr_gain, 2),
        "ssim_gain":     round(ssim_gain, 4)
    }
}
with open(metrics_json_path, "w") as f:
    json.dump(metrics_data, f, indent=2)
print(f"Metrics saved to: {metrics_json_path}")
print_timer("Test Evaluation and Metrics Export", _cell_start)

In [ ]:
_cell_start = time.time()

# Plot Training Curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(range(1, EPOCHS + 1), history["train_loss"], label="Train Loss", marker="o", markersize=4, color="#ef4444")
axes[0].plot(range(1, EPOCHS + 1), history["val_loss"],   label="Val Loss",   marker="s", markersize=4, color="#3b82f6")
axes[0].set_title("L1 + MSE Loss Curve (20 Epochs - CPU)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend()

axes[1].plot(range(1, EPOCHS + 1), history["val_psnr"], label="Val PSNR (dB)", marker="^", markersize=4, color="#10b981")
axes[1].set_title("Validation PSNR Curve (dB)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("PSNR (dB)")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend()

axes[2].plot(range(1, EPOCHS + 1), history["val_ssim"], label="Val SSIM", marker="d", markersize=4, color="#8b5cf6")
axes[2].set_title("Validation SSIM Score Curve", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("SSIM Score")
axes[2].grid(True, linestyle="--", alpha=0.5)
axes[2].legend()

plt.tight_layout()
plots_image_path = os.path.join(OUTPUT_DIR, "loss_psnr_ssim_curves_local.png")
plt.savefig(plots_image_path, dpi=300)
plt.show()

# Visual Restoration Sample Plot
model.eval()
for deg_b, clean_b in test_loader:
    with torch.no_grad():
        res_b = model(deg_b)
    break

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i in range(min(3, deg_b.shape[0])):
    deg_np   = deg_b[i].permute(1, 2, 0).numpy()
    res_np   = res_b[i].permute(1, 2, 0).detach().numpy()
    clean_np = clean_b[i].permute(1, 2, 0).numpy()

    axes[i, 0].imshow(clean_np)
    axes[i, 0].set_title("Original Clean", fontsize=9)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(deg_np)
    axes[i, 1].set_title("Degraded Input", fontsize=9)
    axes[i, 1].axis("off")

    axes[i, 2].imshow(res_np)
    axes[i, 2].set_title("Transformer Restored", fontsize=9)
    axes[i, 2].axis("off")

plt.suptitle("Restormer Transformer - Local CPU - Test Sample Restorations", fontsize=13)
plt.tight_layout()
samples_image_path = os.path.join(OUTPUT_DIR, "test_restoration_samples_local.png")
plt.savefig(samples_image_path, dpi=300)
plt.show()

print_timer("Plots and Visual Restorations", _cell_start)

In [ ]:
_cell_start = time.time()

# Package all results into transformer_restoration_results_local.zip
ZIP_FILE_PATH = os.path.join(r"C:\Users\202511026\OneDrive\Desktop\HK\Image Restoration Project\backend\transformer", "transformer_restoration_results_local.zip")

files_to_zip = [
    (MODEL_SAVE_PATH,    "transformer_model.pth"),
    (metrics_json_path,  "training_metrics_local.json"),
    (plots_image_path,   "loss_psnr_ssim_curves_local.png"),
    (samples_image_path, "test_restoration_samples_local.png")
]

with zipfile.ZipFile(ZIP_FILE_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for src_path, arcname in files_to_zip:
        if os.path.exists(src_path):
            zipf.write(src_path, arcname=arcname)
            print(f"Added to Zip: {arcname}")
        else:
            print(f"Skipped (not found): {arcname}")

print("\n======================================================")
print("LOCAL RESULTS ZIP ARCHIVE CREATED SUCCESSFULLY")
print(f"Zip Path: {ZIP_FILE_PATH}")
print(f"Size: {os.path.getsize(ZIP_FILE_PATH) / (1024 * 1024):.2f} MB")
print("Contains: transformer_model.pth, training_metrics_local.json,")
print("          loss_psnr_ssim_curves_local.png, test_restoration_samples_local.png")
print("======================================================")

print_timer("Zip Packaging and Export", _cell_start)